In [111]:
import pandas as pd
import numpy as np
import pyemu
import yaml
import math
from scipy.special import exp1
import plotly.graph_objects as go

In [112]:
# set up path
target_dir = r"C:\Python\Personal\proj6\codes\sandbox"
rel_t_d = os.path.relpath(target_dir, os.getcwd()) 
os.chdir(rel_t_d)

In [113]:
##########################
### pit-centric domain ###
##########################

# number of wells
n_wells = 2

# number of individuals in the population in the decision variable space
num_reals = 150

# bounds of coordinate system
x_min, x_max = -50, 50
y_min, y_max = -50, 50

# wells cannot be within this many units of the center
no_go_radius = 10.0

# range of pumping rates [m3/day]
q_min, q_max = 6.5, 65

# aquifer properties
T_val = 100.0 # [m2/day]
S_val = 0.001 # [-]
t_eval_val = 365.0 # [day]

######################
### parameter list ###
######################

# build parameter data frame
par_names = []
for i in range(1, n_wells + 1):
    par_names.extend([f"well_{i}_x", f"well_{i}_y", f"well_{i}_q"])

if "Xc" not in par_names:
    par_names.append("Xc")

#######################
### initialize pest ###
#######################

# initialize an empty PEST control file (v2)
pst = pyemu.pst_utils.generic_pst(par_names=par_names)
pdf = pst.parameter_data

###########################
### populate pest files ###
###########################

# randomize compliance point y-position
Yc = np.random.choice([y_min, y_max])

# configure well position metadata
well_pars = [p for p in par_names if "well" in p]
pdf.loc[well_pars, ["partrans", "parchglim", "pargp"]] = ["none", "factor", "well_pos"]
pdf.loc[well_pars, ["parlbnd", "parubnd"]] = [x_min, x_max]

# configure pumping rate metadata
well_q_pars = [p for p in par_names if "_q" in p]
pdf.loc[well_q_pars, ["partrans", "parchglim", "pargp"]] = ["none", "factor", "pumping_rate"]
pdf.loc[well_q_pars, "parlbnd"] = q_min
pdf.loc[well_q_pars, "parubnd"] = q_max

# configure compliance point metadata
pdf.loc["Xc", ["partrans", "parchglim", "pargp"]] = ["none", "factor", "compliance"]
pdf.loc["Xc", ["parlbnd", "parubnd"]] = [x_min, x_max]

###################
### random draw ###
###################

# initialize within constraints
valid_data = []
while len(valid_data) < num_reals:
    # generate random values for all parameters at once
    row = np.random.uniform(0, 1, len(par_names)) # start with 0-1 scale
    
    # scale each parameter to its specific bounds
    scaled_row = []
    for p_name, val in zip(par_names, row):
        lb = pdf.loc[p_name, "parlbnd"]
        ub = pdf.loc[p_name, "parubnd"]
        scaled_row.append(lb + val * (ub - lb))
    
    scaled_row = np.array(scaled_row)
    
    # no-go zone check
    # the first 2*n_wells are X and Y
    # skip Q for the distance check
    well_coords = []
    for i in range(1, n_wells + 1):
        well_coords.append([scaled_row[par_names.index(f"well_{i}_x")], 
                            scaled_row[par_names.index(f"well_{i}_y")]])
    
    distances = np.linalg.norm(well_coords, axis=1)
    if np.all(distances > no_go_radius):
        valid_data.append(scaled_row)

pop = pyemu.ParameterEnsemble(pst=pst, df=pd.DataFrame(valid_data, columns=par_names))


# generate random ensemble for PESTPP-MOU
np.random.seed(np.random.randint(1, 100000))
pop = pyemu.ParameterEnsemble(pst=pst, df=pd.DataFrame(valid_data, columns=par_names))

# select a random realization from the initial population by picking one index name at random from the ensemble
Yc = np.random.choice([y_min, y_max])
selected_real = np.random.choice(pop.index)
pst.parameter_data.loc[par_names, "parval1"] = pop.loc[selected_real, par_names].values

#################################
### center of mass of pumping ###
#################################

# pumping-rate-weighted average of well positions
total_q = sum(pst.parameter_data.loc[f"well_{i}_q", "parval1"] for i in range(1, n_wells + 1))
com_x = sum(pst.parameter_data.loc[f"well_{i}_x", "parval1"] * 
            pst.parameter_data.loc[f"well_{i}_q", "parval1"] for i in range(1, n_wells + 1)) / total_q
com_y = sum(pst.parameter_data.loc[f"well_{i}_y", "parval1"] * 
            pst.parameter_data.loc[f"well_{i}_q", "parval1"] for i in range(1, n_wells + 1)) / total_q

# radial distance from CoM to compliance point and pit (targets)
xc_val = pst.parameter_data.loc["Xc", "parval1"]
dist_com_to_compliance = np.sqrt((com_x - xc_val)**2 + (com_y - Yc)**2)
dist_com_to_pit = np.sqrt(com_x**2 + com_y**2)

#########################################
### generate domain configuration yml ###
#########################################

theis_cfg = {
    'T': T_val, 
    'S': S_val, 
    't_eval': t_eval_val,
    'well_at_compliance': {'Q': float(total_q), 'radial_dist': float(dist_com_to_compliance)},
    'well_at_pit': {'Q': float(total_q), 'radial_dist': float(dist_com_to_pit)}
}

with open("config.yml", "w") as f:
    yaml.dump(theis_cfg, f)

# calculate drawdowns using the logic from run_theis_forward_dd.py
def get_s(Q, r):
    u = (r**2 * S_val) / (4.0 * T_val * t_eval_val)
    return (Q / (4.0 * math.pi * T_val)) * exp1(u)

s_compliance = get_s(total_q, dist_com_to_compliance)
s_pit = get_s(total_q, dist_com_to_pit)

###########################
### objective functions ###
###########################

# min Xc
new_obs = {"obsnme": "obj_min_xc", "obsval": 0.0, "weight": 1.0, "obgnme": "min_xc_gp"}
for col, val in new_obs.items():
    pst.observation_data.loc["obj_min_xc", col] = val

# min total Q
new_obs_q = {"obsnme": "obj_min_q", "obsval": 0.0, "weight": 0.1, "obgnme": "min_q_gp"}
for col, val in new_obs_q.items():
    pst.observation_data.loc["obj_min_q", col] = val

########################
### write and record ###
########################

# set control data for PEST control file (v2)
pst.control_data.noptmax = 0

# write PEST control file (v2)
pst_filename = "fwd_model.pst"
pst.write(pst_filename, version = 2)

# record to external file in the current directory
pop.to_csv("initial_pop.csv")

######################
### output summary ###
######################

print(f"Realization: {selected_real}")

# well summary table
well_data = [{
    "Well": f"{i}",
    "X": pst.parameter_data.loc[f"well_{i}_x", "parval1"],
    "Y": pst.parameter_data.loc[f"well_{i}_y", "parval1"],
    "Q (m3/d)": pst.parameter_data.loc[f"well_{i}_q", "parval1"]
} for i in range(1, n_wells + 1)]

# append CoM as a final row
well_data.append({
    "Well": "CoM",
    "X": com_x,
    "Y": com_y,
    "Q (m3/d)": total_q
})

# drawdown summary table
drawdown_data = [
    {"Target": "Compliance Point", "Radius (m)": dist_com_to_compliance, "Drawdown (m)": s_compliance},
    {"Target": "Pit Center", "Radius (m)": dist_com_to_pit, "Drawdown (m)": s_pit}
]

# display well table
display(pd.DataFrame(well_data).style.format({
    "X": "{:.2f}", 
    "Y": "{:.2f}", 
    "Q (m3/d)": "{:.2f}"
}).hide(axis='index'))

# display drawdown table
display(pd.DataFrame(drawdown_data).style.format({
    "Radius (m)": "{:.2f}", 
    "Drawdown (m)": "{:.2f}"
}).hide(axis='index'))

noptmax:0, npar_adj:7, nnz_obs:3
Realization: 147


Well,X,Y,Q (m3/d)
1,23.56,-42.56,51.86
2,-41.48,18.60,39.99
CoM,-4.76,-15.93,91.85


Target,Radius (m),Drawdown (m)
Compliance Point,73.25,0.70
Pit Center,16.63,0.92


In [114]:
##############
### set up ###
##############

fig = go.Figure()

# access the underlying dataframe using the internal _df attribute
df = pop._df

######################################
### two well example visualization ###
######################################

# domain
fig.add_shape(type="rect", x0=x_min, y0=y_min, x1=x_max, y1=y_max, line=dict(color="black"), opacity=0.1)

# no-go area
fig.add_shape(type="circle", x0=-no_go_radius, y0=-no_go_radius, x1=no_go_radius, y1=no_go_radius, line=dict(color="black", dash="dash"))

# targets with drawdown labels
fig.add_trace(go.Scatter(x=[0], y=[0], mode='markers+text', text=[f"Open Pit<br>s: {s_pit:.2f}m"],
                         marker=dict(color='black', size=10, symbol='hexagon'), name="Open Pit", textposition="top center"))

fig.add_trace(go.Scatter(x=[xc_val], y=[Yc], mode='markers+text', text=[f"Compliance Point<br>s: {s_compliance:.2f}m"],
                         marker=dict(color='yellow', size=10, symbol='star', line=dict(width=1, color='black')), 
                         name="Compliance Point", textposition="top center" if Yc == y_min else "bottom center"))

# well ensembles
fig.add_trace(go.Scatter(x=df["well_1_x"], y=df["well_1_y"], mode='markers', marker=dict(color='lightskyblue', opacity=0.2), name="Well 1 Ensemble"))
fig.add_trace(go.Scatter(x=df["well_2_x"], y=df["well_2_y"], mode='markers', marker=dict(color='lightcoral', opacity=0.2), name="Well 2 Ensemble"))

# selected wells
fig.add_trace(go.Scatter(x=[pst.parameter_data.loc["well_1_x", "parval1"]], y=[pst.parameter_data.loc["well_1_y", "parval1"]],
                         mode='markers+text', text=["Well 1"], marker=dict(color='dodgerblue', size=10), name="Well 1 Selected", textposition="top center"))
fig.add_trace(go.Scatter(x=[pst.parameter_data.loc["well_2_x", "parval1"]], y=[pst.parameter_data.loc["well_2_y", "parval1"]],
                         mode='markers+text', text=["Well 2"], marker=dict(color='crimson', size=10, symbol='square'), name="Well 2 Selected", textposition="top center"))

# CoM and distance lines
fig.add_trace(go.Scatter(x=[com_x], y=[com_y], mode='markers+text', text=["CoM"], marker=dict(color='black', size=12, symbol='x'), name="Pumping Center of Mass", textposition="top center"))
fig.add_trace(go.Scatter(x=[com_x, xc_val, None, com_x, 0], y=[com_y, Yc, None, com_y, 0], 
                         mode='lines', line=dict(color='grey', width=1, dash='dot'), showlegend=False))

fig.update_layout(
    xaxis_title="X Coordinate", 
    yaxis_title="Y Coordinate", 
    template="plotly_white", 
    width=700, 
    height=700,
    title=(f"Theis Analysis: Realization {selected_real}<br>"
           f"<sup>Aquifer Parameters: T={T_val} m²/d, S={S_val}</sup>")
)

fig.show()